In [1]:
import json
import numpy as np
import pandas as pd
from ipie.analysis.autocorr import reblock_by_autocorr

# 1. Load the raw block energies for the A_1 Acene system
with open("scaling_metrics_Classical_A_5.json", "r") as f:
    e_class = np.array(json.load(f)["results"]["raw_block_energies"])

with open("scaling_metrics_GNN_A_5.json", "r") as f:
    e_gnn = np.array(json.load(f)["results"]["raw_block_energies"])

# 2. Compute the block-by-block difference array
# Flatten it to guarantee a strictly 1D shape (N,) to satisfy IPIE
e_diff = (e_gnn - e_class).flatten()

# 3. Pass the strictly 1D numpy array to IPIE
# We pass 'name' to label the output columns, with a try/except just in case 
# your specific IPIE version handles kwargs differently.
try:
    df_diff_ac = reblock_by_autocorr(e_diff, name="E_Diff", verbose=0)
except TypeError:
    df_diff_ac = reblock_by_autocorr(e_diff, verbose=0)

print(f"IPIE Output Columns: {df_diff_ac.columns.tolist()}")

# 4. Safely extract using positional indexing (Row 0, Columns 0 and 1)
# This completely ignores column names, guaranteeing it won't throw a KeyError
mean_diff = float(df_diff_ac.iloc[0, 0])
error_diff = float(df_diff_ac.iloc[0, 1])

print(f"ΔE (GNN - Classical) = {mean_diff*627.5095 :.3f} ± {error_diff*627.5095 :.3f} kcal/mol")

ValueError: operands could not be broadcast together with shapes (71,) (201,) 